# 01 — Dataset

Loads the ASVspoof 2019 LA and 2021 DF datasets, parses their protocol files, and exposes a unified `ASVspoofDataset` for all splits.
Audio is padded or trimmed to exactly **64 600 samples** (4.04 s at 16 kHz).

## Dataset Overview

| Split | Audio dir | Protocol |
|-------|-----------|----------|
| `train` | `data/LA/ASVspoof2019_LA_train/flac/` | `ASVspoof2019.LA.cm.train.trn.txt` |
| `dev` | `data/LA/ASVspoof2019_LA_dev/flac/` | `ASVspoof2019.LA.cm.dev.trl.txt` |
| `la_eval` | `data/LA/ASVspoof2019_LA_eval/flac/` | `ASVspoof2019.LA.cm.eval.trl.txt` |
| `df_eval` | `data/DF/flac/` | `data/DF/keys/DF/CM/trial_metadata.txt` |

**Label convention:** 0 = bonafide, 1 = spoof

**LA protocol** (space-separated, one line per utterance):
```
SPEAKER_ID  FILE_NAME  -  SYSTEM_ID  KEY
```

**DF protocol** (space-separated, **no header line**):
```
col 0: speaker   col 1: file_name   col 2: codec   col 3: source   col 4: system_id   col 5: key
```

## Imports & Paths

In [ ]:
from pathlib import Path
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.utils.data import Dataset
import torchaudio

torchaudio.set_audio_backend("soundfile")

ROOT           = Path(r"C:\Ali\Programming\ai_genrated_audio_detection")
TARGET_SAMPLES = 64600

assert ROOT.exists(), f"Project root not found: {ROOT}"

AUDIO_DIRS = {
    "train"  : ROOT / "data/LA/ASVspoof2019_LA_train/flac",
    "dev"    : ROOT / "data/LA/ASVspoof2019_LA_dev/flac",
    "la_eval": ROOT / "data/LA/ASVspoof2019_LA_eval/flac",
    "df_eval": ROOT / "data/DF/flac",
}

PROTOCOL_FILES = {
    "train"  : ROOT / "data/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.train.trn.txt",
    "dev"    : ROOT / "data/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.dev.trl.txt",
    "la_eval": ROOT / "data/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.eval.trl.txt",
    "df_eval": ROOT / "data/DF/keys/DF/CM/trial_metadata.txt",
}

print(f"ROOT = {ROOT}\n")
for name in PROTOCOL_FILES:
    p = PROTOCOL_FILES[name]
    print(f"  [{name:8s}] protocol exists={p.exists()}  audio_exists={AUDIO_DIRS[name].exists()}")

## Protocol Parser

In [ ]:
def parse_la_protocol(path):
    """
    LA format (space-sep): SPEAKER  FILE_NAME  -  SYSTEM_ID  KEY
    Returns {utt_id: (label:int, attack_id:str)}
    """
    label_map = {}
    with open(path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            uid, atk, key = parts[1], parts[3], parts[4]
            label_map[uid] = (0 if key == "bonafide" else 1, atk)
    return label_map


def parse_df_protocol(path):
    """
    DF format (space-sep, NO header):
      col 0 speaker | col 1 file_name | col 2 codec | col 3 source | col 4 system_id | col 5 key
    Returns {utt_id: (label:int, attack_id:str)}
    """
    label_map = {}
    with open(path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 6:
                continue
            uid = parts[1]          # e.g. DF_E_2000011
            atk = parts[4]          # system / attack id
            key = parts[5]          # 'spoof' or 'bonafide'
            label_map[uid] = (0 if key == "bonafide" else 1, atk)
    return label_map


print("Parsers defined.")

## Dataset Class

In [ ]:
class ASVspoofDataset(Dataset):
    """
    Unified dataset for ASVspoof 2019 LA and 2021 DF.

    subset: 'train' | 'dev' | 'la_eval' | 'df_eval'

    __getitem__ returns:
        waveform  : Tensor [1, 64600]
        label     : int  (0=bonafide, 1=spoof)
        utt_id    : str
        attack_id : str
    """

    def __init__(self, subset: str):
        assert subset in AUDIO_DIRS, f"subset must be one of {list(AUDIO_DIRS)}"
        self.audio_dir    = AUDIO_DIRS[subset]
        protocol_path     = PROTOCOL_FILES[subset]

        if subset == "df_eval":
            if not self.audio_dir.exists() or not protocol_path.exists():
                print(f"DF data not found — skipping.\n"
                      f"  audio    : {self.audio_dir}\n"
                      f"  protocol : {protocol_path}")
                self.items = []
                return
            label_map = parse_df_protocol(protocol_path)
        else:
            label_map = parse_la_protocol(protocol_path)

        self.items = [
            (str(self.audio_dir / f"{uid}.flac"), lbl, uid, atk)
            for uid, (lbl, atk) in label_map.items()
        ]

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        path, label, uid, atk = self.items[idx]
        try:
            wav, _ = torchaudio.load(path)
        except Exception:
            try:
                import av
                container = av.open(path)
                stream = container.streams.audio[0]
                chunks = [frame.to_ndarray() for frame in container.decode(stream)]
                container.close()
                wav = torch.from_numpy(np.concatenate(chunks, axis=1).astype(np.float32))
            except Exception:
                wav = torch.zeros(1, TARGET_SAMPLES)
        if wav.size(0) > 1:
            wav = wav.mean(0, keepdim=True)
        return self._fix_length(wav), label, uid, atk

    @staticmethod
    def _fix_length(wav: torch.Tensor) -> torch.Tensor:
        n = wav.size(1)
        if n < TARGET_SAMPLES:
            wav = wav.repeat(1, TARGET_SAMPLES // n + 1)
        return wav[:, :TARGET_SAMPLES]


print("ASVspoofDataset defined.")

## Verification

In [ ]:
print("=== Train split ===")
train_ds = ASVspoofDataset("train")
print(f"Samples : {len(train_ds)}")

wav, lbl, uid, atk = train_ds[0]
print(f"Shape   : {wav.shape}   label={lbl}   utt={uid}   attack={atk}")

cnt = Counter(item[1] for item in train_ds.items)
print(f"Labels  : bonafide={cnt[0]:,}   spoof={cnt[1]:,}")

bona_idx  = next(i for i, it in enumerate(train_ds.items) if it[1] == 0)
spoof_idx = next(i for i, it in enumerate(train_ds.items) if it[1] == 1)
wb, _, uid_b, _ = train_ds[bona_idx]
ws, _, uid_s, _ = train_ds[spoof_idx]

fig, axes = plt.subplots(1, 2, figsize=(14, 3))
axes[0].plot(wb[0].numpy(), linewidth=0.4)
axes[0].set_title(f"Bonafide — {uid_b}")
axes[0].set_xlabel("Sample")
axes[1].plot(ws[0].numpy(), linewidth=0.4, color="tomato")
axes[1].set_title(f"Spoof — {uid_s}")
axes[1].set_xlabel("Sample")
plt.tight_layout()
out = ROOT / "results/la/sample_waveforms.png"
out.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(out, dpi=120)
plt.show()
print(f"Saved to {out}")

In [ ]:
print("=== DF eval split ===")
df_ds = ASVspoofDataset("df_eval")
if len(df_ds) > 0:
    print(f"Samples : {len(df_ds):,}")
    wav, lbl, uid, atk = df_ds[0]
    print(f"Shape   : {wav.shape}   label={lbl}   utt={uid}   attack={atk}")
    cnt = Counter(item[1] for item in df_ds.items)
    print(f"Labels  : bonafide={cnt[0]:,}   spoof={cnt[1]:,}")
else:
    print("DF dataset not available.")